In [1]:
import urllib.request

url = (
    "https://raw.githubusercontent.com/rasbt/"
    "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
    "the-verdict.txt"
)
file_path = "the-verdict.txt"
urllib.request.urlretrieve(url, file_path)

('the-verdict.txt', <http.client.HTTPMessage at 0x10a090710>)

In [2]:
# load the veredict
import re

with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

preprocessed_text = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed_text = [item.strip() for item in preprocessed_text if item.strip()]

In [3]:
all_words = sorted(set(preprocessed_text))
vocab = {word: i for i, word in enumerate(all_words)}
print("This are the first 10 words of the vocabulary:")
for i in range(10):
    print(f"{i}: {all_words[i]}")

This are the first 10 words of the vocabulary:
0: !
1: "
2: '
3: (
4: )
5: ,
6: --
7: .
8: :
9: ;


In [4]:
# Exercice:
# Implement a tokenizer class with arguments vocab
# The class needs to include two functions: encode and decode


class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {value: key for key, value in vocab.items()}

    def encode(self, text):

        tokens = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        tokens = [item.strip() for item in tokens if item.strip()]
        return [self.str_to_int[word] for word in tokens]

    def decode(self, tokens):

        text = " ".join([self.int_to_str[word] for word in tokens])
        # remove spaces before the special characters
        text = re.sub(r'\s+([,.:;?_!"()\'])', r"\1", text)
        return text


tokenizer = SimpleTokenizerV1(vocab)
text = """It's the last he painted, you know," 
           Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print(ids)
print(tokenizer.decode(ids))


[56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]
It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


In [5]:
# Extend the vocabulary with two extra tokens:

all_words.extend(["<|unk|>", "<|endoftext|>"])
vocab = {word: i for i, word in enumerate(all_words)}


# Now we readapt the tokenizer
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {value: key for key, value in vocab.items()}

    def encode(self, text):

        tokens = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        tokens = [item.strip() for item in tokens if item.strip()]
        tokens = [item if item in self.str_to_int else "<|unk|>" for item in tokens]
        return [self.str_to_int[word] for word in tokens]

    def decode(self, tokens):

        text = " ".join([self.int_to_str[word] for word in tokens])
        # remove spaces before the special characters
        text = re.sub(r'\s+([,.:;?_!"()\'])', r"\1", text)
        return text


text1 = "Hello. do you like tea?"
text2 = "In the sunlit terraces of the palace."
text = text1 + " <|endoftext|> " + text2

tokenizer = SimpleTokenizerV2(vocab)
ids = tokenizer.encode(text)
print(ids)
print(tokenizer.decode(ids))


[1130, 7, 355, 1126, 628, 975, 10, 1131, 55, 988, 956, 984, 722, 988, 1130, 7]
<|unk|>. do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.


In [6]:
vocab["<|endoftext|>"]

1131

#  Byte Pair Encoding

In [8]:
# %pip install tiktoken

In [9]:
import tiktoken

In [10]:
tokenizer = tiktoken.get_encoding("gpt2")